# 03 — Data Preprocessing

**Input:** `data/processed/groundwater_clean_2018_2020.csv` (from `02_Data_Cleaning`)

Turn the clean table into modelling-ready features and the target label.

1. **Derive the drinking-water risk target** from BIS 10500:2012 acceptable
   limits: count how many parameters a sample exceeds, then bin the count into
   `Safe` (0) / `Moderate` (1–2) / `High` (3+).
2. **Select feature columns** (ion chemistry) and report their missingness.
3. **Median-impute chemistry features** *for the ML matrix only* — note this is
   a modelling convenience local to this file; the cleaned CSV keeps `NaN`.
4. **Scale** features (standardisation) and keep an unscaled copy.

**Output:** `data/processed/groundwater_ml_ready.csv`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN = ROOT / "data" / "processed" / "groundwater_clean_2018_2020.csv"
OUT_DIR = ROOT / "data" / "processed"

df = pd.read_csv(CLEAN)
print("Loaded:", df.shape)
df.head()

Loaded: (1106, 27)


,year,season,sno,district,mandal,village,lat,lon,gwl,ph,...,so4,na,k,ca,mg,th,sar,rsc,irrigation_class,rsc_class
0,2018,post_monsoon,1,Adilabad,Adilabad,Adilabad,19.668300,78.524700,5.09,8.28,...,46.0,49.0,4.0,48.0,38.896,279.934211,1.273328,-1.198684,C2S1,P.S.
1,2018,post_monsoon,2,Adilabad,Bazarhatnur,Bazarhatnur,19.458888,78.350833,5.10,8.29,...,68.0,42.0,5.0,56.0,63.206,399.893092,0.913166,-3.397862,C3S1,P.S.
2,2018,post_monsoon,3,Adilabad,Gudihatnoor,Gudihatnoor,19.525555,78.512222,4.98,7.69,...,44.0,45.0,2.0,24.0,38.896,219.934211,1.319284,-0.398684,C2S1,P.S.
3,2018,post_monsoon,4,Adilabad,Jainath,Jainath,19.730555,78.640000,5.75,8.09,...,35.0,27.0,1.0,32.0,19.448,159.967105,0.928155,0.000658,C2S1,P.S.
4,2018,post_monsoon,5,Adilabad,Narnoor,Narnoor,19.495665,78.852654,2.15,8.21,...,280.0,298.0,5.0,56.0,92.378,519.843750,5.682664,-4.396875,C4S2,P.S.


## 1. BIS 10500:2012 drinking-water risk label

Acceptable limits (mg/L unless noted). `pH` is a range; the rest are upper bounds. Missing values don't count as exceedances.

In [2]:
BIS_LIMITS = {
    "ph_low": 6.5, "ph_high": 8.5,
    "tds": 500, "th": 200, "cl": 250,
    "f": 1.0, "no3": 45, "so4": 200, "ca": 75, "mg": 30,
}

def count_exceedances(row):
    n = 0
    if pd.notna(row["ph"]) and not (BIS_LIMITS["ph_low"] <= row["ph"] <= BIS_LIMITS["ph_high"]):
        n += 1
    for key in ["tds","th","cl","f","no3","so4","ca","mg"]:
        v = row[key]
        if pd.notna(v) and v > BIS_LIMITS[key]:
            n += 1
    return n

df["bis_exceedances"] = df.apply(count_exceedances, axis=1)

def risk_label(n):
    if n == 0: return "Safe"
    if n <= 2: return "Moderate"
    return "High"

df["drinking_risk"] = df["bis_exceedances"].apply(risk_label)
print(df["drinking_risk"].value_counts().to_string())

drinking_risk
High        841
Moderate    176
Safe         89


## 2. Feature selection

Ion-chemistry features for predicting drinking risk. (EC / SAR are excluded here to keep this target's features independent of the irrigation-hazard definitions used elsewhere in the project.)

In [3]:
FEATURES = ["ph","tds","co3","hco3","cl","f","no3","so4","na","k","ca","mg","th"]
print("Feature missingness:")
print(df[FEATURES].isna().sum().to_string())

Feature missingness:
ph        1
tds       0
co3     160
hco3      0
cl        0
f         0
no3       0
so4       0
na        0
k         0
ca        0
mg        0
th        0


## 3. Median-impute features (ML matrix only)

The cleaned CSV keeps `NaN`; here we fill for the model matrix so estimators can consume it. Medians are robust to the skew typical of chemistry data.

In [4]:
X = df[FEATURES].copy()
medians = X.median(numeric_only=True)
X = X.fillna(medians)
print("Remaining NaNs after impute:", int(X.isna().sum().sum()))

Remaining NaNs after impute: 0


## 4. Scale

Standardise for scale-sensitive models; keep both scaled and unscaled versions.

In [5]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=[f"{c}_z" for c in FEATURES], index=X.index)

ml = pd.concat([
    df[["year","district","bis_exceedances","drinking_risk"]].reset_index(drop=True),
    X.reset_index(drop=True),
    X_scaled.reset_index(drop=True),
], axis=1)
print("ML-ready shape:", ml.shape)
ml.head()

ML-ready shape: (1106, 30)


,year,district,bis_exceedances,drinking_risk,ph,tds,co3,hco3,cl,f,...,hco3_z,cl_z,f_z,no3_z,so4_z,na_z,k_z,ca_z,mg_z,th_z
0,2018,Adilabad,2,Moderate,8.28,476.80,0.0,220.0,60,0.44,...,-0.597892,-0.669591,-0.865088,-0.336689,0.106131,-0.657005,-0.184979,-0.553204,-0.310241,-0.527881
1,2018,Adilabad,4,High,8.29,589.44,0.0,230.0,80,0.56,...,-0.523851,-0.569844,-0.711785,0.262650,0.513042,-0.718522,-0.136691,-0.429432,0.309979,-0.073181
2,2018,Adilabad,2,Moderate,7.69,326.40,0.0,200.0,30,0.66,...,-0.745973,-0.819210,-0.584032,-0.344956,0.069139,-0.692158,-0.281554,-0.924521,-0.310241,-0.755308
3,2018,Adilabad,0,Safe,8.09,270.08,0.0,160.0,10,0.58,...,-1.042137,-0.918957,-0.686234,-0.661158,-0.097325,-0.850345,-0.329842,-0.800748,-0.806417,-0.982611
4,2018,Adilabad,7,High,8.21,1485.44,0.0,300.0,340,2.56,...,-0.005564,0.726859,1.843265,0.551985,4.434187,1.531261,-0.136691,-0.429432,1.054243,0.381487


## Save ML-ready table

In [6]:
out = OUT_DIR / "groundwater_ml_ready.csv"
ml.to_csv(out, index=False)
print("Wrote", out.relative_to(ROOT), "|", ml.shape)

Wrote data\processed\groundwater_ml_ready.csv | (1106, 30)
